# Bootstrap pareado por evento — Full224 vs Full384

Reconstrucción reproducible del análisis estadístico utilizado para comparar las dos resoluciones de entrada.

La unidad de remuestreo es el **evento**, no el frame. Esto evita tratar miles de imágenes temporalmente correlacionadas como observaciones independientes.


## Diseño

- Test común: **3 761 imágenes**.
- Eventos: **50**.
- Umbral común: **0.50**.
- Réplicas bootstrap: **10 000**.
- Comparación pareada: Full224 y Full384 se evalúan sobre la misma muestra de eventos en cada réplica.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

N_BOOTSTRAP = 10_000
SEED = 42
THRESHOLD = 0.50


## Formato esperado

El análisis parte de predicciones emparejadas a nivel de imagen con, como mínimo:

```text
event_key,y_true,p_224,p_384
```


In [ ]:
def metrics(y_true, prob, threshold=THRESHOLD):
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()

    return {
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "auc": roc_auc_score(y_true, prob),
        "fpr": fp / (fp + tn) if (fp + tn) else np.nan,
    }


In [ ]:
def paired_event_bootstrap(df, n_bootstrap=N_BOOTSTRAP, seed=SEED):
    rng = np.random.default_rng(seed)
    events = df["event_key"].drop_duplicates().to_numpy()
    rows = []

    for _ in range(n_bootstrap):
        sampled_events = rng.choice(events, size=len(events), replace=True)

        # Mantiene multiplicidad cuando un evento aparece varias veces
        parts = [df[df["event_key"] == event] for event in sampled_events]
        sample = pd.concat(parts, ignore_index=True)

        m224 = metrics(sample["y_true"].to_numpy(), sample["p_224"].to_numpy())
        m384 = metrics(sample["y_true"].to_numpy(), sample["p_384"].to_numpy())

        rows.append({
            metric: m384[metric] - m224[metric]
            for metric in m224
        })

    return pd.DataFrame(rows)


## Resultados observados conservados

| Métrica | Full224 | Full384 | Δ 384−224 |
|---|---:|---:|---:|
| Balanced Accuracy | 0.74503 | 0.79894 | +0.05391 |
| Precision | 0.82200 | 0.90195 | +0.07995 |
| Recall | 0.66891 | 0.69057 | +0.02166 |
| F1 | 0.73760 | 0.78223 | +0.04463 |
| AUC | 0.79884 | 0.83860 | +0.03976 |
| FPR | 0.17885 | 0.09269 | −0.08616 |


## Intervalos bootstrap del experimento

| Métrica | Δ observado | IC 95 % |
|---|---:|---:|
| Balanced Accuracy | +0.053906 | [+0.011823, +0.094373] |
| Precision | +0.079950 | [+0.029951, +0.132518] |
| Recall | +0.021655 | [−0.052802, +0.091838] |
| F1 | +0.044633 | [−0.006443, +0.093335] |
| AUC | +0.039764 | [−0.001604, +0.080197] |
| FPR | −0.086156 | [−0.149079, −0.029908] |

Los intervalos excluyen cero a favor de Full384 para **Balanced Accuracy, Precision y FPR**. Para Recall, F1 y AUC, el experimento no aporta un intervalo del 95 % que excluya cero.


## Interpretación

La conclusión más sólida no es que 384 mejore todas las métricas, sino que en esta comparación controlada:

- aumenta la Balanced Accuracy;
- aumenta la Precision;
- reduce de forma marcada la FPR;
- los falsos positivos bajan de **301 a 156**.

Esta interpretación respeta la incertidumbre observada en el bootstrap por evento.
